In [11]:
import pandas as pd 

df = pd.read_csv('../data/interview_cleaned_for_training.csv')

print("Shape:", df.shape)
print()
print("Columns:", df.columns.tolist())
print()
print("Dtypes:")
print(df.dtypes)
print()
print("Missing values:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())
print()
df.head()

Shape: (2000000, 7)

Columns: ['Question_ID', 'Academic_Field', 'User_Specialization', 'Targeted_User_Prompt', 'Humanlike_AI_Response', 'Eye_Movement_Telemetry', 'Proctoring_Cheating_Label']

Dtypes:
Question_ID                  str
Academic_Field               str
User_Specialization          str
Targeted_User_Prompt         str
Humanlike_AI_Response        str
Eye_Movement_Telemetry       str
Proctoring_Cheating_Label    str
dtype: object

Missing values:
Question_ID                  0
Academic_Field               0
User_Specialization          0
Targeted_User_Prompt         0
Humanlike_AI_Response        0
Eye_Movement_Telemetry       0
Proctoring_Cheating_Label    0
dtype: int64

Duplicate rows: 0



,Question_ID,Academic_Field,User_Specialization,Targeted_User_Prompt,Humanlike_AI_Response,Eye_Movement_Telemetry,Proctoring_Cheating_Label
0,INT_SPEC_0000001,Human Resources,Talent Acquisition,Can you help me understand: What exactly is ov...,"Well, Overfitting happens when your model memo...",Stable gaze profile. Eyes focused on monitor w...,Normal
1,INT_SPEC_0000002,Human Resources,Talent Acquisition,"Hey, What's the correct procedure if a patient...",Sure! Stop the transfusion immediately! Discon...,Stable gaze profile. Eyes focused on monitor w...,Normal
2,INT_SPEC_0000003,Engineering,Electrical Engineering,"Hey, What exactly is overfitting and how do we...",Sure! Overfitting happens when your model memo...,Stable gaze profile. Eyes focused on monitor w...,Normal
3,INT_SPEC_0000004,Health Sciences,Nursing,What are the primary diagnostic markers you lo...,"Basically, You look closely at fasting blood g...",Frequent lateral eye movements towards the bot...,Suspected Cheating
4,INT_SPEC_0000005,Computer Science,Software Engineering,What is the best way to handle a salary negoti...,"Well, Focus on the total compensation package ...",Stable gaze profile. Eyes focused on monitor w...,Normal


In [12]:
print("Class distribution:")
print(df['Proctoring_Cheating_Label'].value_counts())
print(df['Proctoring_Cheating_Label'].value_counts(normalize=True))
print()

print("Idadi ya makundi ya kipekee:")
print("Academic_Field:", df['Academic_Field'].nunique())
print("User_Specialization:", df['User_Specialization'].nunique())
print()

print("Uhusiano Academic_Field -> User_Specialization:")
print(df.groupby('Academic_Field')['User_Specialization'].nunique())
print()

print("Mifano ya Eye_Movement_Telemetry (values za kipekee):")
print("Idadi ya maandishi ya kipekee:", df['Eye_Movement_Telemetry'].nunique())
print(df['Eye_Movement_Telemetry'].value_counts().head(10))

Class distribution:
Proctoring_Cheating_Label
Normal                1000069
Suspected Cheating     999931
Name: count, dtype: int64
Proctoring_Cheating_Label
Normal                0.500035
Suspected Cheating    0.499966
Name: proportion, dtype: float64

Idadi ya makundi ya kipekee:
Academic_Field: 5
User_Specialization: 9

Uhusiano Academic_Field -> User_Specialization:
Academic_Field
Business & Finance    2
Computer Science      2
Engineering           2
Health Sciences       2
Human Resources       1
Name: User_Specialization, dtype: int64

Mifano ya Eye_Movement_Telemetry (values za kipekee):
Idadi ya maandishi ya kipekee: 7
Eye_Movement_Telemetry
Stable gaze profile. Eyes focused on monitor workspace with natural micro-saccades. [Metrics: Max Gaze Deviation=2.4°, Off-screen Cumulative Time=0.4s]                                                333554
Stable gaze profile. Eyes focused on monitor workspace with natural micro-saccades. [Metrics: Max Gaze Deviation=6.7°, Off-screen Cumul

In [13]:
crosstab = pd.crosstab(df['Eye_Movement_Telemetry'], df['Proctoring_Cheating_Label'])
print(crosstab)

Proctoring_Cheating_Label                           Normal  Suspected Cheating
Eye_Movement_Telemetry                                                        
Frequent lateral eye movements towards the bott...       0              250899
Frequent lateral eye movements towards the bott...       0              250185
Frequent lateral eye movements towards the late...       0              249582
Frequent lateral eye movements towards the top-...       0              249265
Stable gaze profile. Eyes focused on monitor wo...  333205                   0
Stable gaze profile. Eyes focused on monitor wo...  333554                   0
Stable gaze profile. Eyes focused on monitor wo...  333310                   0


In [14]:
# Kutoa namba ya Gaze Deviation kutoka ndani ya maandishi
df['Gaze_Extracted'] = df['Eye_Movement_Telemetry'].str.extract(r'Max Gaze Deviation=([\d.]+)°').astype(float)

# Kutoa namba ya Off-screen Time kutoka ndani ya maandishi
df['OffScreen_Extracted'] = df['Eye_Movement_Telemetry'].str.extract(r'Off-screen Cumulative Time=([\d.]+)s').astype(float)

print("Idadi ya thamani za kipekee za Gaze_Extracted:", df['Gaze_Extracted'].nunique())
print(df['Gaze_Extracted'].unique())
print()
print("Idadi ya thamani za kipekee za OffScreen_Extracted:", df['OffScreen_Extracted'].nunique())
print(df['OffScreen_Extracted'].unique())

Idadi ya thamani za kipekee za Gaze_Extracted: 7
[ 2.4 11.2 30.6  6.7 45.3 28.6 42.2]

Idadi ya thamani za kipekee za OffScreen_Extracted: 7
[ 0.4  0.1  4.7  0.6  9.  11.6  7.1]


In [15]:
print(df.columns.tolist())

['Question_ID', 'Academic_Field', 'User_Specialization', 'Targeted_User_Prompt', 'Humanlike_AI_Response', 'Eye_Movement_Telemetry', 'Proctoring_Cheating_Label', 'Gaze_Extracted', 'OffScreen_Extracted']


In [16]:
crosstab_gaze = pd.crosstab(df['Gaze_Extracted'], df['Proctoring_Cheating_Label'])
print(crosstab_gaze)

Proctoring_Cheating_Label  Normal  Suspected Cheating
Gaze_Extracted                                       
2.4                        333554                   0
6.7                        333310                   0
11.2                       333205                   0
28.6                            0              249265
30.6                            0              250899
42.2                            0              249582
45.3                            0              250185


MWANZO WA UFUNDISHAJI WA MODEL YETU

In [18]:
# Maktaba za msingi za kuchakata data
import pandas as pd
import numpy as np

# Maktaba za kugawanya data na kupima modeli
from sklearn.model_selection import train_test_split

# Maktaba ya kuona michoro (kuangalia class imbalance kwa jicho)
import matplotlib.pyplot as plt
import seaborn as sns

# Kupakia data
df = pd.read_csv('../data/interview_cleaned_for_training.csv')

# Kuangalia shape (idadi ya rows na columns) kuthibitisha imepakiwa sahihi
print("Shape ya data:", df.shape)
df.head()

Matplotlib is building the font cache; this may take a moment.


Shape ya data: (2000000, 7)


,Question_ID,Academic_Field,User_Specialization,Targeted_User_Prompt,Humanlike_AI_Response,Eye_Movement_Telemetry,Proctoring_Cheating_Label
0,INT_SPEC_0000001,Human Resources,Talent Acquisition,Can you help me understand: What exactly is ov...,"Well, Overfitting happens when your model memo...",Stable gaze profile. Eyes focused on monitor w...,Normal
1,INT_SPEC_0000002,Human Resources,Talent Acquisition,"Hey, What's the correct procedure if a patient...",Sure! Stop the transfusion immediately! Discon...,Stable gaze profile. Eyes focused on monitor w...,Normal
2,INT_SPEC_0000003,Engineering,Electrical Engineering,"Hey, What exactly is overfitting and how do we...",Sure! Overfitting happens when your model memo...,Stable gaze profile. Eyes focused on monitor w...,Normal
3,INT_SPEC_0000004,Health Sciences,Nursing,What are the primary diagnostic markers you lo...,"Basically, You look closely at fasting blood g...",Frequent lateral eye movements towards the bot...,Suspected Cheating
4,INT_SPEC_0000005,Computer Science,Software Engineering,What is the best way to handle a salary negoti...,"Well, Focus on the total compensation package ...",Stable gaze profile. Eyes focused on monitor w...,Normal


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# --- 1. Gawanya data KWANZA (kabla ya encoding) ---
# Tunatumia stratified split ili kuhifadhi uwiano wa 50/50 wa target
X_indices = df.index
train_idx, test_idx = train_test_split(
    X_indices, 
    test_size=0.3, 
    stratify=df['Proctoring_Cheating_Label'], 
    random_state=42
)

train_df = df.loc[train_idx]
test_df = df.loc[test_idx]

print("Training rows:", train_df.shape[0])
print("Testing rows:", test_df.shape[0])

Training rows: 1400000
Testing rows: 600000


In [21]:
# --- One-Hot Encoding: Academic_Field na User_Specialization ---
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)

# FIT kwenye TRAINING data tu, kisha TRANSFORM zote mbili
train_cat_encoded = ohe.fit_transform(train_df[['Academic_Field', 'User_Specialization']])
test_cat_encoded = ohe.transform(test_df[['Academic_Field', 'User_Specialization']])

print("Shape ya category features (train):", train_cat_encoded.shape)
print("Majina ya columns mpya:", ohe.get_feature_names_out())

Shape ya category features (train): (1400000, 14)
Majina ya columns mpya: ['Academic_Field_Business & Finance' 'Academic_Field_Computer Science'
 'Academic_Field_Engineering' 'Academic_Field_Health Sciences'
 'Academic_Field_Human Resources' 'User_Specialization_Civil Engineering'
 'User_Specialization_Corporate Finance'
 'User_Specialization_Data Science & AI'
 'User_Specialization_Digital Marketing'
 'User_Specialization_Electrical Engineering'
 'User_Specialization_Medicine & Surgery' 'User_Specialization_Nursing'
 'User_Specialization_Software Engineering'
 'User_Specialization_Talent Acquisition']


In [22]:
# --- TF-IDF: Targeted_User_Prompt ---
tfidf_prompt = TfidfVectorizer(max_features=500, stop_words='english')
train_prompt_tfidf = tfidf_prompt.fit_transform(train_df['Targeted_User_Prompt'])
test_prompt_tfidf = tfidf_prompt.transform(test_df['Targeted_User_Prompt'])

# --- TF-IDF: Humanlike_AI_Response ---
tfidf_response = TfidfVectorizer(max_features=500, stop_words='english')
train_response_tfidf = tfidf_response.fit_transform(train_df['Humanlike_AI_Response'])
test_response_tfidf = tfidf_response.transform(test_df['Humanlike_AI_Response'])

print("Shape ya prompt TF-IDF (train):", train_prompt_tfidf.shape)
print("Shape ya response TF-IDF (train):", train_response_tfidf.shape)

Shape ya prompt TF-IDF (train): (1400000, 62)
Shape ya response TF-IDF (train): (1400000, 97)


In [23]:
crosstab_prompt = pd.crosstab(train_df['Targeted_User_Prompt'], train_df['Proctoring_Cheating_Label'])
crosstab_prompt['leakage_check'] = crosstab_prompt.max(axis=1) / crosstab_prompt.sum(axis=1)
print(crosstab_prompt.sort_values('leakage_check', ascending=False).head(15))

Proctoring_Cheating_Label                           Normal  \
Targeted_User_Prompt                                         
Can you help me understand: What is the best wa...   15649   
Hey, Can you explain concrete curing and why we...   15721   
Hey, What exactly is overfitting and how do we ...   15318   
Hey, How do you design a behavioral interview q...   15388   
So, What's the difference between SEO and SEM w...   15656   
Quick question: Can you explain concrete curing...   15313   
Quick question: What's the correct procedure if...   15562   
Quick question: How does a transformer step up ...   15805   
So, How does a transformer step up voltage with...   15705   
Can you help me understand: What's the correct ...   15513   
Quick question: What are the primary diagnostic...   15606   
So, Can you explain concrete curing and why we ...   15375   
Can you help me understand: What exactly is ove...   15603   
Can you help me understand: Quick question, how...   15361   
Hey, Wha

In [24]:
crosstab_response = pd.crosstab(train_df['Humanlike_AI_Response'], train_df['Proctoring_Cheating_Label'])
crosstab_response['leakage_check'] = crosstab_response.max(axis=1) / crosstab_response.sum(axis=1)
print("Idadi ya majibu ya kipekee:", train_df['Humanlike_AI_Response'].nunique())
print(crosstab_response.sort_values('leakage_check', ascending=False).head(15))

Idadi ya majibu ya kipekee: 45
Proctoring_Cheating_Label                           Normal  \
Humanlike_AI_Response                                        
Basically, It uses electromagnetic induction. A...   15872   
Basically, SEO is the long game where you optim...   15648   
You ask for highly specific, situational past e...   15480   
Well, Focus on the total compensation package i...   15709   
Well, SEO is the long game where you optimize c...   15587   
It's pretty intuitive! It uses a technique call...   15472   
Focus on the total compensation package includi...   15759   
That's a good one! It uses electromagnetic indu...   15320   
Sure! Focus on the total compensation package i...   15714   
Basically, Overfitting happens when your model ...   15319   
Basically, It's pretty intuitive! It uses a tec...   15620   
Sure! You look closely at fasting blood glucose...   15799   
Basically, Concrete doesn't dry to harden; it u...   15663   
Sure! It uses electromagnetic induction